# BanglaSlumNet — 5-minute Smoke Test
Run this FIRST, before any CU spend. Verifies wiring on 4 synthetic tiles.
**No GPU required. No model downloads.**

In [ ]:
# Cell 1: Repo setup
import subprocess, sys, os
from pathlib import Path

# If running in Colab, mount Drive and clone repo
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/gdrive')
    PROJECT_ROOT = Path('/gdrive/MyDrive/BanglaSlumNet')
    REPO_DIR = PROJECT_ROOT / 'repo'
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/namaray/BanglaSlumNet.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    os.chdir(str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR))
else:
    # Local: assume running from repo root
    REPO_DIR = Path('.')
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Cell 2: Check GPU (warn if < 24 GB)
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | {gb:.1f} GB')
    if gb < 24:
        print('WARNING: Less than 24 GB VRAM. Consider enabling load_in_4bit in config.')
else:
    print('No GPU found — smoke test runs on CPU (models will be slow but wiring still verifiable)')

In [ ]:
# Cell 3: Data layer smoke test (synthetic arrays, no real data needed)
print('Testing data/tiles.py ...')
from src.data.tiles import _smoke_test as tiles_smoke
tiles_smoke()

print('Testing data/augment.py ...')
from src.data.augment import _smoke_test as augment_smoke
augment_smoke()

print('Testing data/socioeconomic.py ...')
from src.data.socioeconomic import _smoke_test as socioec_smoke
socioec_smoke()

In [ ]:
# Cell 4: Model forward-pass smoke test for all 4 configs
print('Testing all 4 model configs ...')
from src.models.banglaslumnet import _smoke_test as model_smoke
model_smoke()

In [ ]:
# Cell 5: SAS-Net smoke test
from src.models.sasnet import _smoke_test as sasnet_smoke
sasnet_smoke()

from src.models.fusion import _smoke_test as fusion_smoke
fusion_smoke()

from src.models.decoder import _smoke_test as decoder_smoke
decoder_smoke()

from src.models.baseline_cnn import _smoke_test as baseline_smoke
baseline_smoke()

In [ ]:
# Cell 6: Visualization smoke test (no real results needed)
from src.viz.plots import _smoke_test as plots_smoke
plots_smoke()

from src.viz.qualitative import _smoke_test as qual_smoke
qual_smoke()

In [ ]:
# Cell 7: Weak-label smoke test
from src.data.weak_labels import _smoke_test as wl_smoke
wl_smoke()

In [ ]:
# Cell 8: Registry and recorder smoke test
import tempfile, json
from src.tracking.registry import RunRegistry
from src.tracking.recorder import ResultsRecorder

with tempfile.TemporaryDirectory() as tmp:
    reg = RunRegistry(f'{tmp}/registry.json')
    reg.register('test_run', 'abc123', 'smoke_test')
    assert not reg.is_done('test_run')
    reg.set_status('test_run', 'done')
    assert reg.is_done('test_run')

    rec = ResultsRecorder(results_dir=tmp)
    rec.record(
        run_id='smoke_run',
        experiment='smoke',
        config={'model': {'config': 'full'}, 'seed': 42},
        metrics={'hc_iou': 0.65, 'precision': 0.70, 'recall': 0.60, 'f1': 0.65},
    )

print('Registry and recorder smoke test passed.')

In [ ]:
# Cell 9: Summary
print('=' * 50)
print('ALL SMOKE TESTS PASSED')
print('Wiring verified. Safe to proceed to BanglaSlumNet_Colab.ipynb')
print('=' * 50)